# Graph-Based 3D Point-Cloud Anomaly Detection

This notebook implements a complete training pipeline for detecting and localizing geometric anomalies in 3D point clouds using Graph Neural Networks.

## Features
- **Multi-scale graph construction**: Fine graph for micro-defects, coarse graph for macro-defects
- **Hierarchical encoder**: EdgeConv/Graph Attention based architecture
- **Self-supervised pretraining**: Masked Patch Modeling + Contrastive Learning
- **Memory bank-based anomaly detection**: Distance to normal prototypes
- **Category-agnostic design**: Works on unseen object types with minimal calibration

## Target: AUROC > 0.70 on unseen categories

## 1. Setup and Installation

In [ ]:
# Install required packages (run once)
# !pip install torch torch-geometric torch-scatter torch-sparse torch-cluster
# !pip install open3d numpy scipy matplotlib plotly tqdm scikit-learn h5py pyyaml

import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import yaml
from tqdm.notebook import tqdm

# Add src to path
sys.path.insert(0, '..')

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Import project modules
from src.data import (
    PointCloudPreprocessor, 
    GraphBuilder, 
    AnomalyShapeNetDataset
)
from src.models import (
    HierarchicalGraphEncoder, 
    AnomalyDetector
)
from src.training import (
    MaskedPatchModeling, 
    ContrastiveLearning, 
    CombinedPretraining,
    AnomalyTrainer,
    CategoryCalibrator
)
from src.training.trainer import custom_collate_fn, create_data_loaders
from src.utils import (
    compute_auroc, 
    compute_pro, 
    visualize_heatmap,
    CategoryMetrics
)

print("All modules imported successfully!")

## 2. Configuration

In [ ]:
# Load configuration
config_path = '../configs/default.yaml'

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Override paths for notebook
config['data']['data_root'] = '../data/Anomaly-ShapeNet'
config['paths']['checkpoint_dir'] = '../checkpoints'
config['paths']['visualization_dir'] = '../visualizations'

# Create directories
for path_key in ['checkpoint_dir', 'visualization_dir', 'log_dir', 'results_dir']:
    Path(config['paths'][path_key]).mkdir(parents=True, exist_ok=True)

print("Configuration loaded:")
print(f"  - Target points: {config['data']['target_points']}")
print(f"  - Encoder type: {config['model']['encoder_type']}")
print(f"  - Embedding dim: {config['model']['out_channels']}")
print(f"  - Batch size: {config['training']['batch_size']}")

## 3. Download Anomaly-ShapeNet Dataset

Download from: https://github.com/Chopper-233/Anomaly-ShapeNet

In [ ]:
# Clone the Anomaly-ShapeNet repository (if not already present)
data_root = Path(config['data']['data_root'])

if not data_root.exists():
    print("Downloading Anomaly-ShapeNet dataset...")
    !git clone https://github.com/Chopper-233/Anomaly-ShapeNet.git {data_root}
else:
    print(f"Dataset already exists at {data_root}")

# List available categories
if data_root.exists():
    categories = [d.name for d in data_root.iterdir() if d.is_dir() and not d.name.startswith('.')]
    print(f"\nAvailable categories ({len(categories)}):")
    for cat in sorted(categories):
        print(f"  - {cat}")

## 4. Data Loading and Preprocessing

In [ ]:
# Initialize preprocessor and graph builder
preprocessor = PointCloudPreprocessor(
    target_points=config['data']['target_points'],
    outlier_nb_neighbors=config['data']['outlier_nb_neighbors'],
    outlier_std_ratio=config['data']['outlier_std_ratio'],
    normal_knn=config['data']['normal_knn']
)

graph_builder = GraphBuilder(
    fine_k=config['graph']['fine_k'],
    coarse_k=config['graph']['coarse_k'],
    num_superpoints=config['graph']['num_superpoints'],
    include_edge_features=config['graph']['include_edge_features']
)

print("Preprocessor initialized:")
print(f"  - Target points: {preprocessor.target_points}")
print(f"  - Normal kNN: {preprocessor.normal_knn}")
print(f"\nGraph builder initialized:")
print(f"  - Fine k: {graph_builder.fine_k}")
print(f"  - Coarse k: {graph_builder.coarse_k}")
print(f"  - Superpoints: {graph_builder.num_superpoints}")

In [ ]:
# Create datasets
# Note: Update data_root path based on your actual dataset location

# For demonstration, we'll check if dataset exists
data_root = Path(config['data']['data_root'])

if data_root.exists():
    # Create training dataset (normal samples only for pretraining)
    train_dataset = AnomalyShapeNetDataset(
        data_root=str(data_root),
        split='train',
        preprocessor=preprocessor,
        graph_builder=graph_builder,
        return_graphs=True,
        point_format=config['data']['point_format']
    )
    
    # Create test dataset
    test_dataset = AnomalyShapeNetDataset(
        data_root=str(data_root),
        split='test',
        preprocessor=preprocessor,
        graph_builder=graph_builder,
        return_graphs=True,
        point_format=config['data']['point_format']
    )
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Test samples: {len(test_dataset)}")
else:
    print(f"Dataset not found at {data_root}")
    print("Please download the Anomaly-ShapeNet dataset first.")
    print("You can use synthetic data for testing (see next cell).")

In [ ]:
# Create synthetic data for testing if real dataset not available
def create_synthetic_pointcloud(n_points=10000, add_anomaly=False):
    """Create a synthetic sphere point cloud with optional anomaly."""
    # Generate sphere
    theta = np.random.uniform(0, 2*np.pi, n_points)
    phi = np.random.uniform(0, np.pi, n_points)
    
    r = 1.0
    x = r * np.sin(phi) * np.cos(theta)
    y = r * np.sin(phi) * np.sin(theta)
    z = r * np.cos(phi)
    
    points = np.stack([x, y, z], axis=1).astype(np.float32)
    
    # Add anomaly (dent or bulge)
    if add_anomaly:
        # Add a bulge in a random region
        center = np.random.randn(3)
        center = center / np.linalg.norm(center)
        
        distances = np.linalg.norm(points - center, axis=1)
        mask = distances < 0.3
        
        # Bulge outward
        direction = points[mask] - np.array([[0, 0, 0]])
        direction = direction / (np.linalg.norm(direction, axis=1, keepdims=True) + 1e-8)
        points[mask] += 0.1 * direction
    
    return points

# Test with synthetic data
print("Testing with synthetic data...")
test_points = create_synthetic_pointcloud(10000, add_anomaly=False)
processed = preprocessor.process(test_points)

print(f"Processed points shape: {processed['points'].shape}")
print(f"Features shape: {processed['features'].shape}")
print(f"Normals shape: {processed['normals'].shape}")

## 5. Visualize Sample Data

In [ ]:
# Visualize a sample point cloud
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 5))

# Normal sample
ax1 = fig.add_subplot(121, projection='3d')
normal_points = create_synthetic_pointcloud(5000, add_anomaly=False)
ax1.scatter(normal_points[:, 0], normal_points[:, 1], normal_points[:, 2], 
            c=normal_points[:, 2], s=0.5, alpha=0.6)
ax1.set_title('Normal Sample')

# Anomaly sample
ax2 = fig.add_subplot(122, projection='3d')
anomaly_points = create_synthetic_pointcloud(5000, add_anomaly=True)
ax2.scatter(anomaly_points[:, 0], anomaly_points[:, 1], anomaly_points[:, 2],
            c=anomaly_points[:, 2], s=0.5, alpha=0.6)
ax2.set_title('Anomaly Sample (with bulge)')

plt.tight_layout()
plt.show()

## 6. Build Model

In [ ]:
# Create the hierarchical encoder
encoder = HierarchicalGraphEncoder(
    in_channels=config['model']['in_channels'],
    hidden_channels=config['model']['hidden_channels'],
    coarse_hidden=config['model']['coarse_hidden'],
    out_channels=config['model']['out_channels'],
    fine_layers=config['model']['fine_layers'],
    coarse_layers=config['model']['coarse_layers'],
    encoder_type=config['model']['encoder_type'],
    k_fine=config['graph']['fine_k'],
    k_coarse=config['graph']['coarse_k'],
    heads=config['model']['heads'],
    dropout=config['model']['dropout'],
    batch_norm=config['model']['batch_norm']
)

# Create anomaly detector
model = AnomalyDetector(
    encoder=encoder,
    embedding_dim=config['model']['out_channels'],
    memory_bank_size=config['anomaly']['memory_bank_size'],
    local_weight=config['anomaly']['local_weight'],
    global_weight=config['anomaly']['global_weight'],
    top_k_percent=config['anomaly']['top_k_percent']
).to(device)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model created!")
print(f"  - Total parameters: {total_params:,}")
print(f"  - Trainable parameters: {trainable_params:,}")
print(f"  - Encoder type: {config['model']['encoder_type']}")
print(f"  - Output dimension: {config['model']['out_channels']}")

## 7. Self-Supervised Pretraining (Stage 1)

In [ ]:
# Create pretraining module
from src.training.pretraining import CombinedPretraining

pretraining_module = CombinedPretraining(
    encoder=model.encoder,
    mask_weight=1.0,
    contrastive_weight=config['training']['pretrain']['contrastive_weight'],
    mask_ratio=config['training']['pretrain']['mask_ratio'],
    patch_size=config['training']['pretrain']['patch_size'],
    projection_dim=config['training']['pretrain']['projection_dim'],
    temperature=config['training']['pretrain']['temperature']
).to(device)

print("Pretraining module created:")
print(f"  - Mask ratio: {config['training']['pretrain']['mask_ratio']}")
print(f"  - Patch size: {config['training']['pretrain']['patch_size']}")
print(f"  - Contrastive weight: {config['training']['pretrain']['contrastive_weight']}")

In [ ]:
# Create trainer
trainer = AnomalyTrainer(
    model=model,
    device=device,
    learning_rate=config['training']['pretrain']['learning_rate'],
    weight_decay=config['training']['pretrain']['weight_decay'],
    save_dir=config['paths']['checkpoint_dir']
)

print("Trainer initialized")

In [ ]:
# Create data loaders (use synthetic data for demo)
from torch.utils.data import Dataset, DataLoader

class SyntheticDataset(Dataset):
    """Synthetic dataset for demonstration."""
    def __init__(self, n_samples=100, n_points=5000, preprocessor=None, graph_builder=None):
        self.n_samples = n_samples
        self.n_points = n_points
        self.preprocessor = preprocessor or PointCloudPreprocessor(target_points=n_points)
        self.graph_builder = graph_builder or GraphBuilder()
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        # Create random synthetic point cloud
        is_anomaly = np.random.random() > 0.8  # 20% anomalies
        points = create_synthetic_pointcloud(self.n_points, add_anomaly=is_anomaly)
        
        # Preprocess
        processed = self.preprocessor.process(points)
        
        # Build graphs
        features = torch.from_numpy(processed['features'])
        pts = torch.from_numpy(processed['points'])
        normals = torch.from_numpy(processed['normals'])
        
        graphs = self.graph_builder.build_hierarchical_graph(pts, features, normals)
        
        return {
            'points': pts,
            'features': features,
            'normals': normals,
            'fine_graph': graphs['fine'],
            'coarse_graph': graphs['coarse'],
            'assignments': graphs['assignments'],
            'label': 1 if is_anomaly else 0,
            'category': 'synthetic'
        }

# Create synthetic datasets for demo
train_dataset_demo = SyntheticDataset(
    n_samples=50, 
    n_points=2000,  # Smaller for faster demo
    preprocessor=preprocessor,
    graph_builder=graph_builder
)

val_dataset_demo = SyntheticDataset(
    n_samples=10,
    n_points=2000,
    preprocessor=preprocessor,
    graph_builder=graph_builder
)

train_loader = DataLoader(
    train_dataset_demo,
    batch_size=2,
    shuffle=True,
    collate_fn=custom_collate_fn,
    num_workers=0  # Set to 0 for notebook compatibility
)

val_loader = DataLoader(
    val_dataset_demo,
    batch_size=2,
    shuffle=False,
    collate_fn=custom_collate_fn,
    num_workers=0
)

print(f"Created data loaders:")
print(f"  - Training batches: {len(train_loader)}")
print(f"  - Validation batches: {len(val_loader)}")

In [ ]:
# Run pretraining (reduced epochs for demo)
PRETRAIN_EPOCHS = 5  # Use more epochs (100+) for real training

print(f"Starting pretraining for {PRETRAIN_EPOCHS} epochs...")
print("(Increase PRETRAIN_EPOCHS for better results)")

history = trainer.pretrain(
    train_loader=train_loader,
    val_loader=val_loader,
    pretraining_module=pretraining_module,
    epochs=PRETRAIN_EPOCHS,
    warmup_epochs=1,
    log_interval=1,
    save_interval=5,
    early_stopping_patience=10
)

print("\nPretraining complete!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
axes[0].plot(history['train_loss'], label='Train Loss')
if history['val_loss']:
    axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Progress')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Learning rate
axes[1].plot(history['learning_rate'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Learning Rate Schedule')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{config['paths']['visualization_dir']}/training_history.png", dpi=150)
plt.show()

## 8. Calibration with Normal Samples (Stage 2)

In [ ]:
# Create normal-only dataset for calibration
class NormalOnlyDataset(Dataset):
    """Dataset with only normal samples for calibration."""
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset
        # Get only normal sample indices
        self.indices = []
        for i in range(len(base_dataset)):
            sample = base_dataset[i]
            if sample['label'] == 0:  # Normal
                self.indices.append(i)
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        return self.base_dataset[self.indices[idx]]

# For demo, create a fresh normal dataset
normal_dataset = SyntheticDataset(
    n_samples=30,  # 10-30 normal samples as specified
    n_points=2000,
    preprocessor=preprocessor,
    graph_builder=graph_builder
)

# Override to be all normal
normal_dataset_items = []
for i in range(len(normal_dataset)):
    item = normal_dataset[i]
    item['label'] = 0  # Force normal
    normal_dataset_items.append(item)

normal_loader = DataLoader(
    normal_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=custom_collate_fn,
    num_workers=0
)

print(f"Normal samples for calibration: {len(normal_dataset)}")

In [ ]:
# Calibrate the anomaly detector
trainer.calibrate(
    normal_loader=normal_loader,
    n_prototypes=config['anomaly']['memory_bank_size']
)

print("Calibration complete!")
print(f"Memory bank size: {model.memory_bank.shape if model.memory_bank is not None else 'None'}")

## 9. Evaluation

In [ ]:
# Create test dataset
test_dataset_demo = SyntheticDataset(
    n_samples=20,
    n_points=2000,
    preprocessor=preprocessor,
    graph_builder=graph_builder
)

test_loader = DataLoader(
    test_dataset_demo,
    batch_size=1,
    shuffle=False,
    collate_fn=custom_collate_fn,
    num_workers=0
)

# Evaluate
metrics = trainer.evaluate(test_loader, compute_per_point=False)

print("\n" + "="*50)
print("Evaluation Results")
print("="*50)
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")
print("="*50)

## 10. Inference and Visualization

In [ ]:
# Run inference on a single sample
sample = test_dataset_demo[0]

result = trainer.inference(
    data=sample['fine_graph'],
    coarse_data=sample['coarse_graph'],
    assignments=sample['assignments']
)

print(f"Sample label: {'Anomaly' if sample['label'] == 1 else 'Normal'}")
print(f"Object score: {result['object_score'].item():.4f}")
print(f"Threshold: {model.get_threshold():.4f}")
print(f"Prediction: {'Anomaly' if result['object_score'].item() > model.get_threshold() else 'Normal'}")

In [ ]:
# Visualize anomaly heatmap
points = sample['points'].numpy()
heatmap = result['heatmap'].numpy()

fig = plt.figure(figsize=(14, 5))

# Original point cloud
ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(points[:, 0], points[:, 1], points[:, 2],
            c=points[:, 2], s=1, alpha=0.6)
ax1.set_title('Original Point Cloud')

# Anomaly heatmap
ax2 = fig.add_subplot(132, projection='3d')
scatter = ax2.scatter(points[:, 0], points[:, 1], points[:, 2],
                      c=heatmap, cmap='jet', s=1, alpha=0.6)
ax2.set_title('Anomaly Heatmap')
fig.colorbar(scatter, ax=ax2, shrink=0.6)

# Score distribution
ax3 = fig.add_subplot(133)
ax3.hist(heatmap, bins=50, alpha=0.7)
ax3.axvline(x=np.percentile(heatmap, 95), color='r', linestyle='--', label='95th percentile')
ax3.set_xlabel('Anomaly Score')
ax3.set_ylabel('Count')
ax3.set_title('Score Distribution')
ax3.legend()

plt.tight_layout()
plt.savefig(f"{config['paths']['visualization_dir']}/anomaly_heatmap.png", dpi=150)
plt.show()

## 11. Save Model

In [ ]:
# Save the trained model
trainer.save_checkpoint('final_model.pth')
print(f"Model saved to {config['paths']['checkpoint_dir']}/final_model.pth")

# Save calibration data
calibrator = CategoryCalibrator(model, device)
calibrator.calibrate_category('synthetic', normal_loader)
calibrator.save_calibration(f"{config['paths']['checkpoint_dir']}/calibration.json")

## 12. Cross-Category Evaluation (Optional)

For real evaluation, use different categories for training and testing.

In [ ]:
# Example cross-category evaluation
# This would be used with the real Anomaly-ShapeNet dataset

def cross_category_evaluation(model, categories, data_root, preprocessor, graph_builder):
    """Perform cross-category evaluation."""
    metrics_tracker = CategoryMetrics()
    
    for test_cat in categories:
        print(f"\nEvaluating on category: {test_cat}")
        train_cats = [c for c in categories if c != test_cat]
        
        # Load training data (normal samples from other categories)
        # ... pretrain and calibrate ...
        
        # Load test data from held-out category
        # ... evaluate and collect metrics ...
        
        pass  # Implement based on actual dataset structure
    
    return metrics_tracker.compute()

print("Cross-category evaluation can be performed with the real Anomaly-ShapeNet dataset.")
print("See the training script for full implementation.")

## Summary

This notebook demonstrated:

1. **Data Preprocessing**: Point cloud normalization, feature extraction (normals, curvature, roughness)
2. **Graph Construction**: Multi-scale graphs for micro and macro defect detection
3. **Model Architecture**: Hierarchical Graph Neural Network with EdgeConv/Attention
4. **Self-Supervised Pretraining**: Masked Patch Modeling + Contrastive Learning
5. **Calibration**: Memory bank-based normal modeling
6. **Evaluation**: Object-level and point-level anomaly detection

For production use:
- Download the full Anomaly-ShapeNet dataset
- Increase training epochs (100+)
- Use full resolution point clouds
- Perform proper cross-category validation

Target: **AUROC > 0.70** on unseen categories